# VINO omar/ -- Full Training (B1 x {NH, MR, AB}, B2 x {NH, MR, AB})
**Colab / T4 GPU**

Runs the exact same training loop as `VINO_Hyperelasticity.py` (Adam,
1000 epochs, unchanged `utils/fno_utils.train_fno`) for all 6 cases. The
only geometry-specific code lives in `Practical_Examples/omar/` -- the FNO
architecture, optimizer, and training loop are untouched upstream code.

After: **Runtime > Change runtime type > T4 GPU**, then run cells in order
starting from Cell 1a. One manual step is unavoidable: Cell 1a
(`condacolab.install()`) restarts the kernel itself partway through
setup (installing conda requires it), which breaks a single uninterrupted
"Run all" -- Colab will likely show "Your session crashed for an unknown
reason" right after that cell, which is expected. Just continue to
Cell 1b once it reconnects; everything after that runs straight through
with no further manual steps (assuming the repo is public, or you've
filled in a token in Cell 2).


## Cell 1a - Install conda (condacolab)

FEniCSx isn't reliably pip-installable, and the old `fem-on-colab` static
installer script breaks whenever Google bumps Colab's base Python image
(it did, on a real run against Python 3.12: `dolfinx` stayed
un-importable even after a real `Runtime > Restart session`). `condacolab`
+ `conda-forge` is the same method that was verified to work end to end
in development, and doesn't depend on a separately-maintained script.

**`condacolab.install()` restarts the kernel itself as part of installing
conda.** Colab will likely show "Your session crashed for an unknown
reason" right after this cell runs -- that is expected here, not a real
failure. Just continue to Cell 1b once the kernel reconnects (no manual
`Runtime > Restart` needed for this specific step).

Must be the first thing installed in a fresh runtime -- if you already
ran pip installs in this session, do `Runtime > Restart session` and
`Runtime > Run all` from scratch first.


In [ ]:
try:
    import dolfinx
    print(f'FEniCSx already installed (dolfinx {dolfinx.__version__}) -- skipping condacolab')
except ImportError:
    !pip install -q condacolab
    import condacolab
    condacolab.install()


## Cell 1b - Install FEniCSx (conda-forge) + JAX/deps (pip)

Run this once the kernel has reconnected after Cell 1a (or immediately,
if Cell 1a printed "already installed" and skipped condacolab).


In [ ]:
try:
    import dolfinx
    print(f'FEniCSx already installed (dolfinx {dolfinx.__version__})')
except ImportError:
    !mamba install -y -q -c conda-forge fenics-dolfinx mpich
    import dolfinx
    print(f'FEniCSx installed (dolfinx {dolfinx.__version__})')

!pip install -q "jax[cuda12]" -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html
!pip install -q flax optax tqdm matplotlib scipy geomdl
!pip install -q torch --index-url https://download.pytorch.org/whl/cpu
print('Done - continue to Cell 2')


## Cell 2 - Clone the repo

If the repo is private, paste a GitHub personal access token as the
value of `GITHUB_TOKEN` below (Settings > Developer settings > Personal
access tokens). Leave it as `""` if the repo is public -- **never type
`<TOKEN>` or any `<...>` placeholder into a shell command**: `<` and `>`
are bash redirection operators, so `https://<TOKEN>@...` makes bash try
to read stdin from a file literally named `TOKEN` and fails before git
even runs (that's the `TOKEN: No such file or directory` error).

Always does a clean re-clone (removes any previous `/content/OMAR` first)
so a broken partial clone from an earlier failed run can't linger.


In [ ]:
import os
import shutil
import sys

# cd out of /content/OMAR *before* possibly deleting it -- deleting the
# process's current directory out from under it causes
# "getcwd: cannot access parent directories" on any subsequent relative
# path operation (os.getcwd(), os.chdir('.'), etc).
os.chdir('/content')

GITHUB_TOKEN = ""  # <-- paste your token between the quotes if the repo is private; leave "" if public
BRANCH = "claude/claude-code-question-d307wp"
REPO_URL = (f"https://{GITHUB_TOKEN}@github.com/suhibamro/omar.git" if GITHUB_TOKEN
            else "https://github.com/suhibamro/omar.git")

if os.path.exists('/content/OMAR'):
    shutil.rmtree('/content/OMAR')

!git clone -b {BRANCH} {REPO_URL} /content/OMAR

WORK_DIR = '/content/OMAR/Practical_Examples'
if not os.path.isdir(WORK_DIR):
    raise SystemExit(
        'ERROR: clone failed -- /content/OMAR/Practical_Examples does not exist.\n'
        'Scroll up to the "git clone" output above for the actual error.\n'
        'Common cause: the repo is private and GITHUB_TOKEN is still "".'
    )

os.chdir(WORK_DIR)
sys.path.insert(0, WORK_DIR)
os.makedirs('./data', exist_ok=True)
os.makedirs('./results', exist_ok=True)

utils_ok = os.path.isdir(os.path.join(WORK_DIR, 'utils'))
omar_ok = os.path.isdir(os.path.join(WORK_DIR, 'omar'))
if utils_ok and omar_ok:
    print('Clone OK: utils/ and omar/ both found under Practical_Examples/.')
else:
    print('ERROR: clone looks incomplete.')
    print(f'  utils/ found: {utils_ok}')
    print(f'  omar/  found: {omar_ok}')
    print('Check GITHUB_TOKEN / BRANCH above, then re-run this cell.')

import jax
jax.config.update('jax_enable_x64', True)
print(f'JAX {jax.__version__} | backend={jax.default_backend()} | devices={jax.devices()}')


## Cell 3 - Generate FEM ground-truth data

Generates all 6 `.npz` files from scratch (each is skipped if it already
exists, see `omar/generate_fem_data.py`). Takes ~15-20 min total for all 6
cases x 550 samples on Colab's CPU (FEniCSx doesn't use the GPU; the
*training* below does). Only needs to run once per Colab runtime.


In [ ]:
# Hard gate: refuse to launch generate_fem_data.py (a separate `!python`
# subprocess) unless dolfinx/ufl are actually importable in *this* kernel.
# Running it anyway is what produces a confusing
# "ModuleNotFoundError: No module named 'ufl'" from deep inside the script
# instead of a clear instruction to fix Cell 1a/1b first. (SystemExit is
# raised outside the except block, not chained to it as __context__ --
# some IPython/Colab kernel versions crash internally while formatting a
# chained SystemExit instead of showing the actual message.)
try:
    import dolfinx
    import ufl
    fenicsx_ready = True
    fenicsx_error = None
except ImportError as e:
    fenicsx_ready = False
    fenicsx_error = e

if not fenicsx_ready:
    raise SystemExit(
        f'FEniCSx is not importable in this kernel yet ({fenicsx_error}).\n'
        'Go back to Cell 1a/1b and make sure they completed successfully '
        '(condacolab needs to be the *first* thing installed in a fresh runtime).'
    ) from None

os.chdir(WORK_DIR)
!python omar/generate_fem_data.py


## Cell 4 - Verify setup

In [ ]:
import os

WORK_DIR = '/content/OMAR/Practical_Examples'
DATA_DIR = os.path.join(WORK_DIR, 'data')

EXPECTED_NPZ = [
    'B1_n550_neohookean_64X64.npz', 'B1_n550_mooneyrivlin_64X64.npz', 'B1_n550_arrudaboyce_64X64.npz',
    'B2_n550_neohookean_64X64.npz', 'B2_n550_mooneyrivlin_64X64.npz', 'B2_n550_arrudaboyce_64X64.npz',
]

dir_checks = {
    'utils/': os.path.isdir(os.path.join(WORK_DIR, 'utils')),
    'omar/': os.path.isdir(os.path.join(WORK_DIR, 'omar')),
    'data/': os.path.isdir(DATA_DIR),
}
npz_checks = {f: os.path.isfile(os.path.join(DATA_DIR, f)) for f in EXPECTED_NPZ}

print('=== Setup check ===')
for name, ok in dir_checks.items():
    print(f'  {name:8s} {"OK" if ok else "MISSING"}')
print('  data/ contents:')
for f, ok in npz_checks.items():
    print(f'    {f:40s} {"OK" if ok else "MISSING"}')

code_ok = dir_checks['utils/'] and dir_checks['omar/']
data_ok = dir_checks['data/'] and all(npz_checks.values())

if code_ok and data_ok:
    print('\nAll checks passed -- continue to Cell 5 (training).')
elif code_ok and not data_ok:
    print('\nFEM data is missing/incomplete. Scroll up to Cell 3\'s output for errors, then re-run it.')
    raise SystemExit('Setup check: FEM data missing -- see messages above.')
else:
    print('\nSTOP: repo structure is incomplete. Go back and fix Cell 2 (clone) before continuing.')
    raise SystemExit('Setup check failed -- see messages above.')


## Cell 5 - Train all 6 cases

Reuses `omar/train_B1.py` and `omar/train_B2.py` directly -- same
`utils.fno_utils.train_fno` loop, same `jax.example_libraries.optimizers.adam`,
same `num_epoch = 1000` / `batch_size = 50` / `learning_rate = 0.001` from
`omar/config.py` as the original `VINO_Hyperelasticity.py`. Nothing here
duplicates or reimplements that loop.

`module.run()` returns a dict (`{'params', 'train_losses', 'test_losses'}`)
-- not a tuple -- so don't unpack it positionally if you ever need to
rewrite this cell by hand.

Each case is checkpointed to `./results/{name}_checkpoint.pkl` right after
it finishes, and skipped (loaded from disk instead of retrained) if that
checkpoint already exists. If the runtime disconnects partway through the
~5-hour run, just re-run this cell: finished cases resume instantly from
their checkpoint and only the remaining ones actually retrain.


In [ ]:
import os
import time
import pickle
import omar.train_B1 as train_B1
import omar.train_B2 as train_B2
from omar.config import B1_NH, B1_MR, B1_AB, B2_NH, B2_MR, B2_AB

CASES = [
    ("B1_NH", train_B1, B1_NH, 0), ("B1_MR", train_B1, B1_MR, 1), ("B1_AB", train_B1, B1_AB, 2),
    ("B2_NH", train_B2, B2_NH, 3), ("B2_MR", train_B2, B2_MR, 4), ("B2_AB", train_B2, B2_AB, 5),
]

os.makedirs('./results', exist_ok=True)
results = {}
for name, module, cfg, seed in CASES:
    ckpt_path = f'./results/{name}_checkpoint.pkl'

    if os.path.exists(ckpt_path):
        print(f'\n===== {name}: checkpoint found, loading (skipping training) =====')
        with open(ckpt_path, 'rb') as f:
            results[name] = pickle.load(f)
        continue

    print(f'\n===== Training {name} (1000 epochs, Adam) =====')
    t0 = time.time()
    results[name] = module.run(cfg, seed=seed)
    r = results[name]
    print(f'{name} done in {time.time()-t0:.0f}s | final train={r["train_losses"][-1]:.4f} test={r["test_losses"][-1]:.4f}')

    with open(ckpt_path, 'wb') as f:
        pickle.dump(results[name], f)
    print(f'  checkpoint saved: {ckpt_path}')


## Cell 6 - Loss curves

In [ ]:
import numpy as np
from matplotlib import pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, (name, _, _, _) in zip(axes.flat, CASES):
    r = results[name]
    ax.semilogy(np.abs(r['train_losses']), label='Train')
    ax.semilogy(np.abs(r['test_losses']), label='Test')
    ax.set_title(name)
    ax.legend()
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('./results/loss_curves.png', dpi=150)
plt.show()


## Cell 7 - Error box plots and field comparisons

Same visualizations as Fig. 7/8 in the VINO paper (Eshaghi et al., CMAME
437 (2025) 117785) and the plotting section of the original
`VINO_Hyperelasticity.py`: a box plot of the relative L2 error across
the whole test set, plus exact / predicted / error field comparisons
(X- and Y-displacement) for one representative (median-error) test
sample per case. B2's plots are drawn on the true physical quarter-ring
shape, not the flat (theta, r) computational rectangle.


In [ ]:
import torch
from torch.utils.data import random_split
import omar.postprocessing as pp

DATASET_CLS = {'B1': train_B1.B1Dataset, 'B2': train_B2.B2Dataset}
MODEL_CLS = {'B1': train_B1.FNO2d_B1, 'B2': train_B2.FNO2d_B2}
LOSS_CLS = {'B1': train_B1.B1Loss, 'B2': train_B2.B2Loss}

for name, module, cfg, seed in CASES:
    geometry = name.split('_')[0]
    dataset = DATASET_CLS[geometry](cfg)
    split_generator = torch.Generator().manual_seed(seed)
    _, test_dataset = random_split(dataset, [cfg['fno']['n_train'], cfg['fno']['n_test']], generator=split_generator)
    normalizers = [dataset.normalizer_x, dataset.normalizer_y]

    model = MODEL_CLS[geometry](
        modes1=cfg['fno']['mode1'], modes2=cfg['fno']['mode2'], width=cfg['fno']['width'],
        depth=cfg['fno']['depth'], channels_last_proj=cfg['fno']['channels_last_proj'],
        padding=cfg['fno']['padding'], out_channels=2, model_data=cfg['beam'])
    loss_fn = LOSS_CLS[geometry](model, cfg, normalizers, d=1, p=1, size_average=False)

    print(f'\n===== {name}: field comparison + error box plot =====')
    errors = pp.plot_case_results(name, model, cfg, results[name]['params'], loss_fn, test_dataset, normalizers,
                                  './results', is_b2=(geometry == 'B2'))
    results[name]['test_errors'] = errors
    print(f'{name}: mean test error = {errors.mean():.4f}, median = {np.median(errors):.4f}')


## Cell 8 - Save and download results

In [ ]:
import pickle, shutil
from google.colab import files

for name, _, _, _ in CASES:
    with open(f'./results/params_{name}.pkl', 'wb') as f:
        pickle.dump(results[name]['params'], f)

np.savez('./results/losses.npz', **{
    f'{name}_{split}': results[name][f'{split}_losses']
    for name, _, _, _ in CASES for split in ['train', 'test']
})

np.savez('./results/test_errors.npz', **{
    name: results[name]['test_errors'] for name, _, _, _ in CASES if 'test_errors' in results[name]
})

shutil.make_archive('omar_training_results', 'zip', './results')
files.download('omar_training_results.zip')
print('Saved and downloaded omar_training_results.zip')
